# What are GenAI guardrails?

This offline lab uses an internal HR/IT support assistant to make policy boundaries observable.

## 0. Setup

This lab is entirely offline and uses only synthetic identities, documents, and requests. The module is imported from this lesson directory so the same deterministic controls can be reused in tests.

In [ ]:
import json
import sys
from pathlib import Path

LESSON_DIR = Path.cwd()
sys.path.insert(0, str(LESSON_DIR))
import guardrails_lab as lab

fixtures = LESSON_DIR / 'fixtures'
requests = lab.load_fixtures(fixtures / 'requests.json')
by_id = {request['id']: request for request in requests}
kb_data = lab.load_fixtures(fixtures / 'kb.json')
kb = [lab.Document(**item) for item in kb_data]
policies = lab.default_policies()
print(f'{len(requests)} requests, {len(kb)} documents, {len(policies)} policies')

## 1. Define the operating envelope as data

Policy records make scope, signals, decisions, owners, evidence, exceptions, and outage behavior explicit. Read these records as the operating envelope the application enforces rather than as instructions for a model.

In [ ]:
for policy in policies:
    print({
        'policy_id': policy.policy_id,
        'scope': policy.scope,
        'signal': policy.signal,
        'decision': policy.decision_on_violation.value,
        'owner': policy.owner,
        'evidence': policy.evidence,
        'on_error': policy.on_error.value,
    })

## 2. Input rail: authentication, size, and scope

The input rail can reject unauthenticated or oversized requests and escalate injection-looking text before model inference. It cannot authorize a payroll transaction because it cannot see the eventual tool call. `topic_classifier` and `injection_heuristic` are heuristic stand-ins, not live models.

In [ ]:
for request in (by_id['req-01'], by_id['req-02'], by_id['req-13']):
    identity = lab.Identity(**request['identity'])
    record = lab.guard_input(lab.request_text(request), identity, policies)
    print(request['id'], record.decision.value, record.reasons)

print('Input checks cannot authorize a payroll transaction; execution must do that.')

## 3. Retrieval rail: tenant filter before context

The tenant filter runs before context construction, so an unauthorized business-unit document never reaches the assistant. Note that the poisoned document is quarantined while its provenance is retained in decision metadata.

In [ ]:
for request in (by_id['req-03'], by_id['req-04']):
    identity = lab.Identity(**request['identity'])
    selected = [doc for doc in kb if doc.doc_id in request['retrieved_doc_ids']]
    filtered, records = lab.guard_retrieval(selected, identity, policies)
    print(request['id'], [doc.doc_id for doc in filtered])
    for record in records:
        print(record.decision.value, record.reasons, record.metadata)

## 4. Execution rail: authorization and verification

Execution controls validate tool names, arguments, roles, and approval requirements at the application boundary. The receipt example shows that post-execution verification and reconciliation are separate from pre-execution authorization.

In [ ]:
for request in (by_id['req-05'], by_id['req-07'], by_id['req-08']):
    identity = lab.Identity(**request['identity'])
    call = lab.ToolCall(**request['tool_call'])
    record = lab.guard_tool_call(identity, call, policies)
    print(request['id'], record.decision.value, record.reasons)

ledger = lab.Ledger()
receipt = lab.verify_tool_result(
    lab.ToolCall('read_ticket', {'ticket_id': 'ticket-42'}),
    {'success': True, 'operation_id': 'op-42'},
    ledger,
)
print(receipt.decision.value, ledger.entries)

## 5. Output rail: PII redaction and groundedness

The output rail redacts synthetic PII before text is shown. Groundedness is deterministic here: every answer must cite `[doc-XXX]`, and each citation must be present in the retrieved evidence.

In [ ]:
for request in (by_id['req-10'], by_id['req-11'], by_id['req-12'], by_id['req-15']):
    identity = lab.Identity(**request['identity'])
    selected = [doc for doc in kb if doc.doc_id in request.get('retrieved_doc_ids', [])]
    record = lab.guard_output(request['output_text'], selected, policies)
    print(request['id'], record.decision.value, record.reasons, record.metadata)

## 6. Fail-open versus fail-closed

The topic classifier is fail-open because a low-risk routing error can be monitored without authorizing a sensitive action. The write-path authorization check is fail-closed because an outage must never permit an irreversible payroll side effect.

In [ ]:
topic_outage = dict(lab.DEFAULT_DETECTORS)
topic_outage['topic_classifier'] = lab.failing_detector
identity = lab.Identity('user-2001', 'employee', 'bu-north', True)
open_record = lab.guard_input('What is the weather forecast?', identity, policies, topic_outage)
print('topic outage:', open_record.decision.value, open_record.reasons, open_record.metadata)

write_outage = {'authorization': lab.failing_detector}
write_request = by_id['req-05']
write_identity = lab.Identity(**write_request['identity'])
write_call = lab.ToolCall(**write_request['tool_call'])
closed_record = lab.guard_tool_call(write_identity, write_call, policies, write_outage)
print('write outage:', closed_record.decision.value, closed_record.reasons, closed_record.metadata)

## 7. Reproduce why guardrails fail

The correctly ordered pipeline checks authorization before it can record a side effect. The misordered version records the payroll attempt first, demonstrating why a later output decision cannot undo an action.

In [ ]:
request = by_id['req-05']
identity = lab.Identity(**request['identity'])
safe_ledger = lab.Ledger()
safe = lab.run_pipeline(request, identity, policies, kb, safe_ledger)
unsafe_ledger = lab.Ledger()
unsafe = lab.run_pipeline(request, identity, policies, kb, unsafe_ledger, misordered=True)
print('correct ordering:', safe.terminal.value, safe_ledger.entries)
print('misordered:', unsafe.terminal.value, unsafe_ledger.entries)
print('An output decision cannot undo a side effect that already happened.')

## 8. Decision records and summary table

Compare each observed terminal decision with the fixture label to see how layered rails compose. The final JSON object demonstrates audit-safe reasons, policy IDs, detector versions, correlation IDs, and hashed identities.

In [ ]:
for request in requests:
    identity = lab.Identity(**request['identity'])
    result = lab.run_pipeline(request, identity, policies, kb, lab.Ledger())
    print(request['id'], request['expected_decision'], result.terminal.value, result.deciding_rail)

sample_request = by_id['req-12']
sample = lab.run_pipeline(sample_request, lab.Identity(**sample_request['identity']), policies, kb, lab.Ledger())
print(json.loads(sample.records[-1].to_json()))

## Exercises

1. Add a `PolicyRecord` for an unlisted risk and predict its decision.
2. Flip one rail's `on_error` value and predict the outage result before running.
3. Move the role check into a system prompt string and show that the unauthorized write is no longer blocked.